# ptbr-llm-bench: primeira rodada (modelos abertos)

Runtime: **GPU T4** (Ambiente de execução → Alterar tipo). Roda o Qwen2.5-1.5B base e com o LoRA `pedido-extractor-ptbr-lora` nas 3 tarefas. Sem custo de API.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ptbr-llm-bench'
!mkdir -p {DRIVE}
!test -d /content/ptbr-llm-bench || git clone https://github.com/pedrohbenetton/ptbr-llm-bench /content/ptbr-llm-bench
%cd /content/ptbr-llm-bench
!git pull
!pip install -q -e '.[hf]'

In [ ]:
# Teste rápido (5 itens por tarefa) para ver se tudo carrega antes da rodada completa.
!python -m ptbrbench --model hf:Qwen/Qwen2.5-1.5B-Instruct --limit 5 --out /content/smoke

In [ ]:
# Rodada completa: base vs base + LoRA. Salva no Drive; se já existir, não repete.
!test -f {DRIVE}/rodada2/results.json || python -m ptbrbench \
    --model hf:Qwen/Qwen2.5-1.5B-Instruct \
    --model hf:Qwen/Qwen2.5-1.5B-Instruct@pedrohbenetton/pedido-extractor-ptbr-lora \
    --out {DRIVE}/rodada2

In [ ]:
print(open(f'{DRIVE}/rodada2/report.md', encoding='utf-8').read())

## Diagnóstico: o que cada modelo respondeu de fato

Mostra as respostas brutas de `intencao` e `ambiguidade` (onde os resultados foram estranhos) e conta quantas respostas começam com `{` (JSON no lugar da etiqueta/letra).

In [ ]:
import json, collections
res = json.load(open(f'{DRIVE}/rodada2/results.json', encoding='utf-8'))
for modelo, tarefas in res.items():
    print('==', modelo)
    for tarefa in ('intencao', 'ambiguidade'):
        outs = tarefas[tarefa]['outputs']
        json_like = sum(o.strip().startswith('{') for o in outs.values())
        primeira = collections.Counter(o.strip().split('\n')[0][:20] for o in outs.values())
        print(f'  {tarefa}: {json_like}/{len(outs)} começam com "{{"; mais comuns:', primeira.most_common(5))

## Repontuar a rodada salva (nota rígida e tolerante)

Não roda os modelos de novo: usa as respostas brutas de `rodada2`. Precisa do `git pull` da célula 1 para ter o comando `rescore`.

In [ ]:
!python -m ptbrbench rescore {DRIVE}/rodada2/results.json --out {DRIVE}/rodada2_rescored